#### Imports and Authentication of Google Earth Engine

In [ ]:
import ee
import geemap
import pandas as pd
import geopandas as gpd
import json
import os
import shutil
from functools import reduce
import datetime
import gc
import math # Added for sin/cos calculations

#### First Authenticate Google Earth Engine and Initialize the project

In [ ]:
# Check authentication
ee.Authenticate()
ee.Initialize(project='ee-tsiakirisvasileio-master')
print(ee.String('Hello from the Earth Engine servers!').getInfo())


#### Functions

In [ ]:
def df_to_ee(df):
  features = []
  for i, row in df.iterrows():
    point = ee.Geometry.Point(row["Longitude"], row["Latitude"])
    # Explicitly convert the Python datetime.date object (i) to an ee.Date object string
    # before creating the ee.Date object.
    ee_date_obj = ee.Date(i.isoformat())

    # Handle NaN values in Soil_Moisture by converting them to None
    soil_moisture_value = row["Soil_Moisture"]
    if pd.isna(soil_moisture_value):
        soil_moisture_value = None

    # Calculate DOY_sin and DOY_cos
    doy = row["day_of_year"]
    doy_sin = math.sin(2 * math.pi * doy / 365)
    doy_cos = math.cos(2 * math.pi * doy / 365)

    feat = ee.Feature(None, {"date": ee_date_obj,
                               "Soil_Moisture": soil_moisture_value,
                               "Longitude": row["Longitude"],
                               "Latitude": row["Latitude"],
                               "day_of_year": doy,
                              "dgg_id" : int(row["Grid_Point_ID"]),
                              "source_file": row["source_file"],
                              "DOY_sin": doy_sin,
                              "DOY_cos": doy_cos})

    features.append(feat)
  return ee.FeatureCollection(features)

In [ ]:
def get_rs_data(feature):
    date_prop = feature.get('date')
    date = ee.Date(date_prop) # Convert the date property to an ee.Date object

    # Retrieve hexagonal geometry from the 'hexagons' FeatureCollection
    # Filtering by 'smos_id' which corresponds to the 'dgg_id' property
    DGG_ID = feature.get('dgg_id')
    matched_hexagon = hexagons.filter(ee.Filter.eq('smos_id', DGG_ID)).first()

    # Use the hexagon geometry if available, otherwise fallback to point geometry
    sampling_geom = ee.Geometry(ee.Algorithms.If(matched_hexagon, matched_hexagon.geometry(), feature.geometry()))

    # For NDVI (8-day cadence), find the closest image within +/- 4 days
    ndvi_search_start = date.advance(-4, 'day')
    ndvi_search_end = date.advance(5, 'day') # +1 for exclusive end date

    ndvi_collection_filtered = ndvi_collection.filterDate(ndvi_search_start, ndvi_search_end)

    # Sort by absolute time difference from the feature's date
    ndvi_image = ee.Algorithms.If(
        ndvi_collection_filtered.size().gt(0),
        ndvi_collection_filtered.map(lambda image: image.set('time_diff', ee.Number(image.get('system:time_start')).subtract(ee.Number(date.millis())).abs())).sort('time_diff').first(),
        None # If no image found in the window
    )
    ndvi_scale = ee.Algorithms.If(ndvi_image, ee.Image(ndvi_image).projection().nominalScale(), ee.Number(250)) # Fallback if image not found
    ndvi = ee.Algorithms.If(ndvi_image, ee.Image(ndvi_image).reduceRegion(reducer=ee.Reducer.mean(), geometry=sampling_geom, scale=ndvi_scale, bestEffort=True, maxPixels=1e9).get('NDVI'), None)

    temperature_image = temperature_collection.filterDate(date, date.advance(1, 'day')).first()
    temp_scale = ee.Algorithms.If(temperature_image, ee.Image(temperature_image).projection().nominalScale(), ee.Number(1000))
    temperature = ee.Algorithms.If(temperature_image, ee.Image(temperature_image).reduceRegion(reducer=ee.Reducer.mean(), geometry=sampling_geom, scale=temp_scale, bestEffort=True, maxPixels=1e9).get('LST_Day_1km'), None)

    precipitation_image = precipitation_collection.filterDate(date, date.advance(1, 'day')).first()
    precip_scale = ee.Algorithms.If(precipitation_image, ee.Image(precipitation_image).projection().nominalScale(), ee.Number(5000))
    precipitation = ee.Algorithms.If(precipitation_image, ee.Image(precipitation_image).reduceRegion(reducer=ee.Reducer.mean(), geometry=sampling_geom, scale=precip_scale, bestEffort=True, maxPixels=1e9).get('precipitation'), None)

    # For ET (16-day cadence), find the closest image within +/- 8 days
    et_search_start = date.advance(-8, 'day')
    et_search_end = date.advance(9, 'day') # +1 for exclusive end date

    et_collection_filtered = evapotranspiration_collection.filterDate(et_search_start, et_search_end)

    evapotranspiration_image = ee.Algorithms.If(
        et_collection_filtered.size().gt(0),
        et_collection_filtered.map(lambda image: image.set('time_diff', ee.Number(image.get('system:time_start')).subtract(ee.Number(date.millis())).abs())).sort('time_diff').first(),
        None # If no image found in the window
    )
    et_scale = ee.Algorithms.If(evapotranspiration_image, ee.Image(evapotranspiration_image).projection().nominalScale(), ee.Number(500))
    evapotranspiration = ee.Algorithms.If(evapotranspiration_image, ee.Image(evapotranspiration_image).reduceRegion(reducer=ee.Reducer.mean(), geometry=sampling_geom, scale=et_scale, bestEffort=True, maxPixels=1e9).get('ET'), None)

    # Elevation and land cover are static, no date filter needed
    # Now it's an ImageCollection, so we need to mosaic it to get a single image
    elevation_image = elevation_collection.mosaic()
    elev_scale = elevation_image.projection().nominalScale()
    elevation = elevation_image.reduceRegion(reducer=ee.Reducer.mean(), geometry=sampling_geom, scale=elev_scale, bestEffort=True, maxPixels=1e9).get('DEM') # 'DEM' is the band name for Copernicus DEM

    land_cover_image = land_cover_collection.first()
    land_cover_scale = ee.Algorithms.If(land_cover_image, ee.Image(land_cover_image).projection().nominalScale(), ee.Number(10))
    # For categorical data like land cover, mode is generally more appropriate than mean.
    land_cover = ee.Algorithms.If(land_cover_image, ee.Image(land_cover_image).reduceRegion(reducer=ee.Reducer.mode(), geometry=sampling_geom, scale=land_cover_scale, bestEffort=True, maxPixels=1e9).get('Map'), None) # 'Map' is the band name for WorldCover

    feature = feature.set({
        'NDVI': ndvi,
        'LST_Day_1km': temperature,
        'precipitation': precipitation,
        'ET': evapotranspiration,
        'elevation': elevation,
        'land_cover': land_cover
    })
    return feature

In [ ]:
def chunk_dataframe(df, chunk_size):
    for i in range(0, len(df), chunk_size):
        yield df.iloc[i:i + chunk_size]

In [ ]:
def merge_sm_rs_data(df_SMOS, chunk_size):
    all_chunks = []
    for i, df_chunk in enumerate(chunk_dataframe(df_SMOS, chunk_size)):
        if i % 10 == 0:
            print(f"Processing chunk {i}")

        # 1. Convert chunk to EE FeatureCollection
        smos_fc = df_to_ee(df_chunk)

        # 2. Sample RS data SERVER-SIDE
        sampled_fc = smos_fc.map(get_rs_data)

        # 3. Strip geometry to reduce payload size before bringing it back to local memory
        sampled_fc_no_geom = sampled_fc.select(['.*'], None, False)
        KEEP = ["date", "dgg_id", "Soil_Moisture", "NDVI", "LST_Day_1km", "precipitation", "ET", "elevation", "land_cover", "Latitude", "Longitude", "DOY_sin", "DOY_cos"]
        sampled_fc_small = sampled_fc_no_geom.select(KEEP, None, False)  # retainGeometry=False

        df_rs = geemap.ee_to_df(sampled_fc_small)

        all_chunks.append(df_rs)

    df_final = pd.concat(all_chunks, ignore_index=True)

    # Convert 'date' column from dictionary representation to datetime objects
    if 'date' in df_final.columns and not df_final['date'].empty:
        # Check if the first element is a dictionary, implying EE Date object conversion issue
        if isinstance(df_final['date'].iloc[0], dict):
            # Convert from milliseconds since epoch to datetime
            df_final['date'] = df_final['date'].apply(lambda x: pd.to_datetime(x['value'], unit='ms'))
        else:
            # If not a dict, pd.to_datetime should handle strings or other convertible types
            df_final['date'] = pd.to_datetime(df_final['date'])

    df_final.index = df_final["date"]
    df_final.index.name = "Date"

    return df_final

# Data definitions

In [ ]:
# Dynamic values
start_date, end_date = '2022-01-01', '2024-12-31'

# MODIS
ndvi_collection = ee.ImageCollection('MODIS/061/MOD13A1').filter(ee.Filter.date(start_date, end_date))
temperature_collection = ee.ImageCollection('MODIS/061/MOD11A1').filter(ee.Filter.date(start_date, end_date))
evapotranspiration_collection = ee.ImageCollection('MODIS/061/MOD16A2GF').filter(ee.Filter.date(start_date, end_date))
# CHIRPS
precipitation_collection = ee.ImageCollection('UCSB-CHG/CHIRPS/DAILY').filter(ee.Filter.date(start_date, end_date))

# Constant values
elevation_collection = ee.ImageCollection('COPERNICUS/DEM/GLO30') # Changed from Image to ImageCollection
land_cover_collection = ee.ImageCollection('ESA/WorldCover/v200')

In [ ]:
# Path to the shapefile in Google Drive
shapefile_path = '/content/drive/MyDrive/Lund/ISEA4H9_Hexagons.shp'

HEX_ASSET_ID = "projects/ee-tsiakirisvasileio-master/assets/ISEA4H9_Hexagons"
hexagons = ee.FeatureCollection(HEX_ASSET_ID)

#Dataframe Compiler

In [ ]:
SMOS_csv_data_path = r"/content/drive/MyDrive/SMOS_csvs"
csv_files = [os.path.join(SMOS_csv_data_path, csv) for csv in os.listdir(SMOS_csv_data_path) if csv.endswith(".csv")]

all_processed_dfs = []

for csv_file in csv_files:
    print(f"Processing file: {csv_file}")
    df = pd.read_csv(csv_file)
    df["source_file"] = os.path.basename(csv_file)
    # Assuming 'Date' column exists in CSVs and needs to be parsed
    if 'creation_date' in df.columns:
        df['Date'] = pd.to_datetime(df['creation_date'], format='mixed', errors='coerce')
        df.set_index('Date', inplace=True)

    # Convert the DataFrame's index to date-only by iterating and extracting date part
    df.index = [x.date() for x in df.index]

    # Add day of the year column
    df['day_of_year'] = pd.to_datetime(df.index).dayofyear

    # Process each CSV file individually with merge_sm_rs_data
    # Using a chunk_size that's appropriate for memory usage during EE calls
    processed_df_chunk = merge_sm_rs_data(df, 500)
    all_processed_dfs.append(processed_df_chunk)

# Concatenate all results from individual CSV files into one final DataFrame
final_df = pd.concat(all_processed_dfs, ignore_index=False) # Keep original index if meaningful, or reset if preferred

# The final_df will now contain all data from all CSVs, processed in chunks and concatenated.
# print(final_df.head())
final_df.to_csv('/content/drive/MyDrive/EarthEngine_Exports/Final_complete_SM_dataset-(5-7).csv')

Processing file: /content/drive/MyDrive/SMOS_csvs/SMOS_month_07.csv
Processing chunk 0
Processing chunk 10
Processing chunk 20
Processing chunk 30
Processing chunk 40
Processing chunk 50
Processing chunk 60
Processing chunk 70
Processing chunk 80
Processing chunk 90
Processing chunk 100
Processing chunk 110
Processing chunk 120
Processing chunk 130
Processing chunk 140
Processing chunk 150
Processing chunk 160
Processing chunk 170
Processing chunk 180
Processing chunk 190


Processing chunk 200
Processing chunk 210
Processing chunk 220
Processing chunk 230


Processing chunk 240
Processing chunk 250
Processing file: /content/drive/MyDrive/SMOS_csvs/SMOS_month_06.csv
Processing chunk 0
Processing chunk 10
Processing chunk 20
Processing chunk 30


Processing chunk 40
Processing chunk 50
Processing chunk 60
Processing chunk 70
Processing chunk 80
Processing chunk 90
Processing chunk 100
Processing chunk 110
Processing chunk 120
Processing chunk 130
Processing chunk 140
Processing chunk 150
Processing chunk 160
Processing chunk 170
Processing chunk 180
Processing chunk 190
Processing chunk 200
Processing chunk 210
Processing chunk 220
Processing chunk 230
Processing chunk 240
Processing chunk 250


Processing chunk 260
Processing chunk 270
Processing chunk 280
Processing chunk 290
Processing chunk 300
Processing chunk 310
Processing chunk 320
Processing chunk 330
Processing chunk 340


Processing chunk 350
Processing chunk 360
Processing chunk 370
Processing chunk 380
Processing file: /content/drive/MyDrive/SMOS_csvs/SMOS_month_05.csv
Processing chunk 0
Processing chunk 10
Processing chunk 20
Processing chunk 30
Processing chunk 40
Processing chunk 50
Processing chunk 60
Processing chunk 70
Processing chunk 80
Processing chunk 90
Processing chunk 100
Processing chunk 110
Processing chunk 120
Processing chunk 130
Processing chunk 140
Processing chunk 150
Processing chunk 160
Processing chunk 170
Processing chunk 180
Processing chunk 190
Processing chunk 200
Processing chunk 210
Processing chunk 220
Processing chunk 230
Processing chunk 240


Processing chunk 250
Processing chunk 260
Processing chunk 270
Processing chunk 280
Processing chunk 290
Processing chunk 300
Processing chunk 310
